# Занятие 4. Практика: фильтрация, пропуски и первая модель — АВТОРСКОЕ РЕШЕНИЕ

На прошлом занятии мы разобрали теорию: **булевы маски**, логические операторы `&` `|` `~`,
метод **`.query()`**, `.copy()`, обработку пропусков через `.isna()`, `.dropna()`, `.fillna()`.

Сегодня применяем всё это на реальных данных — и в конце **обучим вашу первую модель**:
за 3 строки построим простейшую линейную регрессию и попробуем что-то предсказать.

**План занятия:**
1. **Разминка на теории** — примеры по каждому инструменту (маски, `.query()`, `.copy()`);
2. **Целевые группы** — выделяем нужных пользователей сложными условиями;
3. **Аномалии** — ищем подозрительные строки жёсткими условиями;
4. **Пропуски** — находим, удаляем и заполняем;
5. **Моя первая модель** — линейная регрессия в 3 строки;
6. **Практика**.


In [1]:
import numpy as np
import pandas as pd

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)
pd.set_option('display.precision', 2)

## Загружаем данные

Тот же датасет про ЕГЭ: 1000 учеников, 16 столбцов.

In [2]:
df = pd.read_csv('data/ege_students.csv', sep=';')
print('Размер:', df.shape)
df.head()

Размер: (1000, 16)


,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
0,1,Роман,М,17,Нижний Новгород,Обычная,Физика,5.4,нет,7,11,5.0,7.9,2.4,47,да
1,2,Арина,Ж,17,Ростов-на-Дону,Обычная,Математика,9.1,нет,8,2,2.0,7.9,3.4,66,да
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да
3,4,Владимир,М,16,Ростов-на-Дону,Обычная,Химия,6.2,да,4,5,1.0,5.0,2.6,58,да
4,5,Дмитрий,М,17,Москва,Обычная,Русский язык,6.5,нет,6,4,2.0,6.8,3.0,61,да


---
## Часть 1. Разминка: булевы маски и `.query()`

### 1.1. Простая маска

Отберём всех, кто сдал ЕГЭ на **90 и выше** — это «стобалльники и почти-стобалльники».

In [3]:
top = df[df['балл_егэ'] >= 90]
print('Учеников с баллом 90+:', len(top))
top.head()

Учеников с баллом 90+: 121


,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
6,7,Роман,М,17,Самара,Лицей,Физика,16.6,нет,8,2,7.0,6.8,3.9,90,да
24,25,Ева,Ж,17,Новосибирск,Обычная,Обществознание,14.6,да,6,2,7.0,10.1,3.9,100,да
26,27,Дмитрий,М,17,Ростов-на-Дону,Обычная,Информатика,15.8,да,5,4,7.0,6.6,3.8,98,да
32,33,Тимофей,М,18,Нижний Новгород,Обычная,Биология,19.3,да,6,0,7.0,5.5,4.3,92,да
44,45,Кирилл,М,17,Санкт-Петербург,Обычная,История,19.4,да,9,3,10.0,6.9,4.3,100,да


### 1.2. Сложное условие через маску: И, ИЛИ, НЕ

Отберём **девочек-лицеисток** с высоким баллом.

In [4]:
mask = (df['пол'] == 'Ж') & (df['тип_школы'] == 'Лицей') & (df['балл_егэ'] >= 80)
print('Подходят:', mask.sum(), 'девочек')
df[mask].head()

Подходят: 33 девочек


,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
16,17,Полина,Ж,17,Казань,Лицей,Химия,9.2,да,9,3,4.0,7.3,3.4,83,да
179,180,София,Ж,17,Самара,Лицей,История,5.6,да,9,2,6.0,5.8,3.4,82,да
212,213,Ксения,Ж,17,Ростов-на-Дону,Лицей,История,7.8,нет,7,1,9.0,6.1,3.6,87,да
231,232,Арина,Ж,18,Екатеринбург,Лицей,Английский язык,11.6,да,8,4,9.0,4.3,3.3,100,да
254,255,Варвара,Ж,18,Новосибирск,Лицей,Русский язык,8.8,да,9,2,9.0,5.8,4.4,100,да


То же самое через **`.query()`** — короче и читаемее:

In [5]:
df.query('пол == "Ж" and тип_школы == "Лицей" and балл_егэ >= 80').head()

,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
16,17,Полина,Ж,17,Казань,Лицей,Химия,9.2,да,9,3,4.0,7.3,3.4,83,да
179,180,София,Ж,17,Самара,Лицей,История,5.6,да,9,2,6.0,5.8,3.4,82,да
212,213,Ксения,Ж,17,Ростов-на-Дону,Лицей,История,7.8,нет,7,1,9.0,6.1,3.6,87,да
231,232,Арина,Ж,18,Екатеринбург,Лицей,Английский язык,11.6,да,8,4,9.0,4.3,3.3,100,да
254,255,Варвара,Ж,18,Новосибирск,Лицей,Русский язык,8.8,да,9,2,9.0,5.8,4.4,100,да


### 1.3. `.isin()` — значение из списка

Ученики из **трёх крупнейших городов**:

In [6]:
cities = ['Москва', 'Санкт-Петербург', 'Казань']
big_city = df[df['город'].isin(cities)]
print('Из трёх городов:', len(big_city))
big_city['город'].value_counts()

Из трёх городов: 372


город
Санкт-Петербург    136
Москва             118
Казань             118
Name: count, dtype: int64

---
## Часть 2. Целевые группы пользователей

В реальной работе часто нужно **выделить группу**, для которой сделаем что-то отдельное:
рассылку, скидку, отдельный анализ. Соберём несколько таких групп.

### Группа A. «Отличники» — балл ≥ 80 и репетитор

Гипотеза: репетитор помогает набрать высокий балл. Отберём тех, кто и **занимался с репетитором**,
и **сдал на 80+**.

In [7]:
group_a = df.query('репетитор == "да" and балл_егэ >= 80').copy()
print('Отличников с репетитором:', len(group_a))
print('Средний балл в группе:', round(group_a['балл_егэ'].mean(), 1))
group_a.head()

Отличников с репетитором: 137
Средний балл в группе: 89.6


,ученик_id,имя,пол,возраст,город,тип_школы,любимый_предмет,часов_подготовки_в_неделю,репетитор,кол_во_пробников,пропусков_уроков,мотивация_1_10,часов_сна,средний_балл_года,балл_егэ,сдал
2,3,Дарья,Ж,17,Самара,Обычная,Русский язык,11.3,да,14,4,7.0,8.2,2.6,80,да
5,6,Милана,Ж,16,Екатеринбург,Обычная,Математика,9.1,да,4,4,6.0,5.4,3.3,81,да
16,17,Полина,Ж,17,Казань,Лицей,Химия,9.2,да,9,3,4.0,7.3,3.4,83,да
23,24,Ева,Ж,17,Новосибирск,Обычная,Биология,12.2,да,8,2,6.0,7.5,3.1,83,да
24,25,Ева,Ж,17,Новосибирск,Обычная,Обществознание,14.6,да,6,2,7.0,10.1,3.9,100,да


> Обратите внимание на **`.copy()`** — мы явно копируем таблицу, чтобы дальше можно было
> добавлять столбцы без предупреждений.

### Группа B. «Трудяги» — много готовятся и мало прогуливают

Условие: `>10` часов подготовки в неделю **И** `<=3` пропусков уроков.

In [8]:
group_b = df.query('часов_подготовки_в_неделю > 10 and пропусков_уроков <= 3').copy()
print('Трудяг:', len(group_b))
print('Их средний балл ЕГЭ:', round(group_b['балл_егэ'].mean(), 1))

Трудяг: 127
Их средний балл ЕГЭ: 87.4


### Группа C. «Столичные» — Москва или Санкт-Петербург

In [9]:
group_c = df[df['город'].isin(['Москва', 'Санкт-Петербург'])].copy()
print('Столичных:', len(group_c))
print('Их средний балл ЕГЭ:', round(group_c['балл_егэ'].mean(), 1))

Столичных: 254
Их средний балл ЕГЭ: 66.9


### Сравним группы

In [10]:
summary = pd.DataFrame({
    'группа':          ['A: репетитор + 80+', 'B: трудяги', 'C: столичные', 'все ученики'],
    'сколько человек': [len(group_a), len(group_b), len(group_c), len(df)],
    'средний балл':    [round(group_a['балл_егэ'].mean(), 1),
                        round(group_b['балл_егэ'].mean(), 1),
                        round(group_c['балл_егэ'].mean(), 1),
                        round(df['балл_егэ'].mean(), 1)],
})
summary

,группа,сколько человек,средний балл
0,A: репетитор + 80+,137,89.6
1,B: трудяги,127,87.4
2,C: столичные,254,66.9
3,все ученики,1000,67.1


**Что видно из таблицы:**
- у групп **A** (репетитор + 80+) и **B** (трудяги) средний балл почти **на 20 выше**, чем у всех;
- группа **C** (столичные) — по среднему почти как все ученики: жить в Москве или Питере
  само по себе высокий балл не гарантирует.

Это и есть **сегментация**: одно и то же поведение (усердно готовиться) даёт результат,
а простая география — нет. В практике будет задание на такое же сравнение.


---
## Часть 3. Поиск аномалий по жёстким условиям

**Аномалия** — это строка, которая выглядит «подозрительно» или «не как все». Обычно её
ищут комбинацией жёстких условий.

### Аномалия 1. Слабый в году, но сдал ЕГЭ на 80+

Средний балл года ≤ 2.5, а на ЕГЭ вдруг 80+. Возможно — списал или необычно готовился.

In [11]:
anomaly1 = df.query('средний_балл_года <= 2.5 and балл_егэ >= 80')
print('Таких аномалий:', len(anomaly1))
anomaly1[['имя', 'город', 'средний_балл_года', 'часов_подготовки_в_неделю', 'балл_егэ']]

Таких аномалий: 4


,имя,город,средний_балл_года,часов_подготовки_в_неделю,балл_егэ
125,Ева,Самара,2.3,10.9,92
147,Тимофей,Москва,2.3,10.3,80
195,Данил,Самара,2.3,12.4,80
705,Милана,Москва,2.5,6.5,80


### Аномалия 2. Ноль часов подготовки

Ученики, которые вообще не готовились. Посмотрим, как они сдали.

In [12]:
zero_prep = df[df['часов_подготовки_в_неделю'] == 0]
print('Не готовились совсем:', len(zero_prep), 'человек')
print('Их средний балл ЕГЭ:', round(zero_prep['балл_егэ'].mean(), 1))
print('Максимум:', zero_prep['балл_егэ'].max())

Не готовились совсем: 64 человек
Их средний балл ЕГЭ: 44.4
Максимум: 78


> **Вывод:** без подготовки максимум — чуть выше среднего. Подтверждает, что подготовка важна.

### Аномалия 3. Крайности через `~`

Найдём тех, кто **НЕ** входит ни в одну из «нормальных» категорий: обычные ученики без
крайностей. Скажем, отбросим и стобалльников, и совсем провалившихся.

In [13]:
normal = df[~((df['балл_егэ'] >= 95) | (df['балл_егэ'] <= 30))]
print('«Обычных» учеников:', len(normal))

«Обычных» учеников: 899


---
## Часть 4. Пропуски — быстрое напоминание

Смотрим, где пропуски и сколько их.

In [14]:
df.isna().sum()

ученик_id                     0
имя                           0
пол                           0
возраст                       0
город                         0
тип_школы                     0
любимый_предмет               0
часов_подготовки_в_неделю     0
репетитор                     0
кол_во_пробников              0
пропусков_уроков              0
мотивация_1_10               25
часов_сна                    40
средний_балл_года             0
балл_егэ                      0
сдал                          0
dtype: int64

Есть пропуски в `мотивация_1_10` и `часов_сна`. Сделаем **чистую копию** таблицы,
в которой пропуски **заполнены медианой** — она устойчивее к выбросам, чем среднее.

In [15]:
df_clean = df.copy()
df_clean['мотивация_1_10'] = df_clean['мотивация_1_10'].fillna(df['мотивация_1_10'].median())
df_clean['часов_сна']     = df_clean['часов_сна'].fillna(df['часов_сна'].median())

print('Пропусков в df_clean:', df_clean.isna().sum().sum())

Пропусков в df_clean: 0


Теперь у нас есть **чистая таблица** `df_clean` без единого `NaN` — с ней можно
безопасно обучать модель.

---
## Часть 5. Моя первая модель — линейная регрессия

**Идея.** У нас есть числа, которые связаны между собой: чем больше ученик готовится,
тем **обычно** выше его балл. Хочется:
1. **измерить** эту связь числом;
2. **предсказывать** балл нового ученика по числу часов подготовки.

### Что вообще такое «линейная регрессия»?

Представьте себе облако точек: по горизонтали — часы подготовки, по вертикали — балл ЕГЭ.
Каждая точка — один ученик. Точки образуют «облако», которое в целом идёт слева-снизу
вправо-вверх (кто больше готовится — у того балл выше).

Линейная регрессия просто **проводит через это облако прямую линию** — так, чтобы она
проходила **как можно ближе ко всем точкам сразу**. Уравнение этой прямой:

$$
\text{балл} \approx a \cdot \text{часы подготовки} + b
$$

где:
- **`a` (наклон)** — **на сколько поднимается линия**, если сдвинуться на 1 час вправо.
  Если `a = 2.5` — значит, каждый дополнительный час подготовки в среднем **добавляет 2.5 балла**.
  Чем круче линия — тем сильнее связь между признаком и результатом.
- **`b` (сдвиг)** — **где линия пересекает вертикальную ось** (при 0 часов подготовки).
  Это как бы «стартовый балл» без всякой подготовки.

Модель сама подбирает `a` и `b` так, чтобы её прямая лучше всего описывала данные.

Посмотрите на картинку — тут всё сразу становится понятно:

![Линейная регрессия](images/linear_regression.png)

**Что на картинке:**

- **синие точки** — 1000 реальных учеников, каждая точка = один человек;
- **красная прямая** — линия, которую подобрала модель. Она проходит «сквозь середину» облака;
- **оранжевая точка** на оси Y — значение `b ≈ 46.8`. Это «стартовый балл»: сюда упирается
  линия при 0 часов подготовки;
- **зелёная стрелка** показывает `a ≈ 2.5`: если сдвинуться вправо на 1 час, линия
  поднимется на 2.5 балла. **Это и есть наклон.**

Теперь, когда мы понимаем, что означают `a` и `b`, давайте обучим такую модель сами.

### 5.1. Обучаем модель в 3 строки

Импортируем модель, создаём её, вызываем `.fit(X, y)` — всё.

In [16]:
from sklearn.linear_model import LinearRegression

X = df_clean[['часов_подготовки_в_неделю']]   # признак (в двойных скобках — это таблица)
y = df_clean['балл_егэ']                       # что предсказываем

model = LinearRegression().fit(X, y)

a = model.coef_[0]
b = model.intercept_
print(f'Наклон (a): {a:.2f}')
print(f'Сдвиг (b):  {b:.2f}')
print(f'Формула модели: балл ≈ {a:.2f} · часы + {b:.2f}')

Наклон (a): 2.52
Сдвиг (b):  47.01
Формула модели: балл ≈ 2.52 · часы + 47.01


Смотрим на числа — и сверяем с картинкой выше:
- `a ≈ 2.53` — та самая зелёная стрелка «+1 час → +2.5 балла»;
- `b ≈ 46.81` — та самая оранжевая точка на оси Y.

Модель — это буквально два числа. Всё, что она умеет, — **умножить и прибавить**.

### 5.2. Делаем прогноз

Спросим у модели: **какой балл получит ученик, готовящийся 5, 10, 15 или 20 часов в неделю?**

In [17]:
hours = pd.DataFrame({'часов_подготовки_в_неделю': [5, 10, 15, 20]})
hours['прогноз_балла'] = model.predict(hours).round(1)
hours

,часов_подготовки_в_неделю,прогноз_балла
0,5,59.6
1,10,72.2
2,15,84.8
3,20,97.4


**Видно** прямую зависимость: каждые +5 часов подготовки → примерно +12–13 баллов.
Именно это и говорит нам наклон `a ≈ 2.5` (5 часов × 2.5 = 12.5 балла).

### 5.3. Модель с несколькими признаками

Настоящая жизнь сложнее одной переменной. Дадим модели **сразу 4 признака** и посмотрим,
какой из них важнее.

In [18]:
features = ['часов_подготовки_в_неделю', 'мотивация_1_10', 'часов_сна', 'средний_балл_года']
X = df_clean[features]
y = df_clean['балл_егэ']

model2 = LinearRegression().fit(X, y)

# посмотрим коэффициенты — «вес» каждого признака
pd.DataFrame({'признак': features, 'коэффициент': model2.coef_.round(2)})

,признак,коэффициент
0,часов_подготовки_в_неделю,1.25
1,мотивация_1_10,2.23
2,часов_сна,0.59
3,средний_балл_года,13.62


**Как это читать:** коэффициент показывает, на сколько баллов изменится прогноз при
увеличении признака на 1 (при неизменных остальных). Это те же `a`, только теперь их 4 —
по одному на каждый признак.

- **средний балл года** — самый мощный признак: +1 балл в году = **+13.5** к ЕГЭ;
- **мотивация** — тоже важна: +1 балл мотивации ≈ +2.3 к ЕГЭ;
- **часы подготовки** — вклад меньше, но всё равно значимый (+1.2 за час);
- **часы сна** — влияние небольшое.

### 5.4. Прогноз для конкретного ученика

Придумаем гипотетического ученика и попросим модель предсказать его балл.

In [19]:
new_student = pd.DataFrame({
    'часов_подготовки_в_неделю': [10],
    'мотивация_1_10':            [7],
    'часов_сна':                 [8],
    'средний_балл_года':         [3.0],
})
pred = model2.predict(new_student)[0]
print('Прогноз балла ЕГЭ:', round(pred, 1))

Прогноз балла ЕГЭ: 72.3


**Это и есть машинное обучение в самом простом виде:**
модель посмотрела на 1000 учеников, нашла связь между признаками и баллом,
и теперь может **прикинуть балл нового ученика**, о котором знает только эти 4 числа.

Дальше в курсе разберём, как оценивать качество модели, как избежать переобучения,
и как строить более сложные модели. Но самое главное — базовый рабочий цикл — вы уже видели:

1. **отфильтровать / очистить** данные;
2. **выбрать признаки** и целевую переменную;
3. **`model.fit(X, y)`** — обучить;
4. **`model.predict(...)`** — предсказать.


---
## Практика — авторское решение

Ниже — 12 заданий на изученный материал. В каждом задании:
- **напишите код** в ячейке `# Ваш код` — код обязателен;
- **выведите результат** через `print()` или последней строкой.

В некоторых заданиях есть пункт **Вывод:** — там нужно короткое предложение о том, что получилось и как это трактовать.

Работаем с таблицей `df`, загруженной выше. Столбцы:
`ученик_id`, `имя`, `пол`, `возраст`, `город`, `тип_школы`, `любимый_предмет`,
`часов_подготовки_в_неделю`, `репетитор`, `кол_во_пробников`, `пропусков_уроков`,
`мотивация_1_10`, `часов_сна`, `средний_балл_года`, `балл_егэ`, `сдал`.

**Максимум за практику — 30 баллов.**


### <font color='DarkOrange'>Задание 1 [2 балла]</font>

Сколько **девочек сдали** ЕГЭ? Отберите тех, у кого `пол == 'Ж'` **И** `сдал == 'да'`,
и посчитайте количество.


In [20]:
print(((df['пол'] == 'Ж') & (df['сдал'] == 'да')).sum())
# Ответ: 487


487


### <font color='DarkOrange'>Задание 2 [2 балла]</font>

Сколько **мальчиков из Москвы** в таблице (`пол == 'М'` **И** `город == 'Москва'`)?
Используйте **`.query()`**.


In [21]:
print(len(df.query('пол == "М" and город == "Москва"')))
# Ответ: 61


61


### <font color='DarkOrange'>Задание 3 [2 балла]</font>

Найдите **средний балл ЕГЭ** среди тех, кто занимался с репетитором (`репетитор == 'да'`).
Округлите до одного знака после запятой.


In [22]:
print(round(df[df['репетитор'] == 'да']['балл_егэ'].mean(), 1))
# Ответ: 68.9


68.9


### <font color='DarkOrange'>Задание 4 [2 балла]</font>

Сколько учеников готовились **больше 10 часов в неделю** **И** **сдали на 80 баллов и выше**?


In [23]:
print(len(df.query('часов_подготовки_в_неделю > 10 and балл_егэ >= 80')))
# Ответ: 172


172


### <font color='DarkOrange'>Задание 5 [2 балла]</font>

Сколько учеников **НЕ из Москвы и НЕ из Санкт-Петербурга**?
Используйте `.isin([...])` и оператор `~`.


In [24]:
print((~df['город'].isin(['Москва', 'Санкт-Петербург'])).sum())
# Ответ: 746


746


### <font color='DarkOrange'>Задание 6 [2 балла]</font>

С помощью **`.query()`** найдите, сколько **лицеистов** (`тип_школы == 'Лицей'`)
с **мотивацией 8 и выше** (`мотивация_1_10 >= 8`).


In [25]:
print(len(df.query('тип_школы == "Лицей" and мотивация_1_10 >= 8')))
# Ответ: 53


53


### <font color='DarkOrange'>Задание 7 [3 балла]</font>

Уберите пропуски в столбце `часов_сна` через **`dropna(subset=[...])`**.
Среди оставшихся возьмите тех, кто **спит 8 часов и больше**, и посчитайте их
**средний балл ЕГЭ** (округлите до одного знака после запятой).


In [26]:
sub = df.dropna(subset=['часов_сна'])
print(round(sub[sub['часов_сна'] >= 8]['балл_егэ'].mean(), 1))
# Ответ: 72.7


72.7


### <font color='DarkOrange'>Задание 8 [3 балла]</font>

**Сравнение: с репетитором vs без.** Помогает ли репетитор?
Посчитайте **средний балл ЕГЭ** отдельно у тех, кто занимался с репетитором (`репетитор == 'да'`),
и у тех, кто нет. **На сколько баллов** средний балл с репетитором **выше**?
Округлите до одного знака.


In [27]:
s_yes = df[df['репетитор'] == 'да']['балл_егэ'].mean()
s_no  = df[df['репетитор'] == 'нет']['балл_егэ'].mean()
print('С репетитором :', round(s_yes, 1))
print('Без репетитора:', round(s_no, 1))
print('Разница       :', round(s_yes - s_no, 1))
# Ответ: 3.4


С репетитором : 68.9
Без репетитора: 65.5
Разница       : 3.4


**Вывод:** разница ~3.4 балла — репетитор чуть помогает, но эффект небольшой.


### <font color='DarkOrange'>Задание 9 [3 балла]</font>

Соберите две группы:
- **«Трудяги»** — `часов_подготовки_в_неделю > 10` **И** `пропусков_уроков <= 3`;
- **«Столичные»** — из Москвы или Санкт-Петербурга (`город in [...]`).

Сколько человек попадают **одновременно в обе** группы (пересечение)?


In [28]:
both = df.query('часов_подготовки_в_неделю > 10 and пропусков_уроков <= 3 '
                'and город in ["Москва", "Санкт-Петербург"]')
print(len(both))
# Ответ: 35


35


### <font color='DarkOrange'>Задание 10 [3 балла]</font>

**Сравнение: лицей vs обычная школа.** Правда ли, что лицеисты сдают лучше?
Найдите **средний балл ЕГЭ** отдельно для `тип_школы == 'Лицей'` и для `тип_школы == 'Обычная'`.
Чему равна **разница** (лицей − обычная), округлённая до одного знака?


In [29]:
m_licey = df[df['тип_школы'] == 'Лицей']['балл_егэ'].mean()
m_obych = df[df['тип_школы'] == 'Обычная']['балл_егэ'].mean()
print('Лицей  :', round(m_licey, 1))
print('Обычная:', round(m_obych, 1))
print('Разница (лицей − обычная):', round(m_licey - m_obych, 1))
# Ответ: -0.7


Лицей  : 66.7
Обычная: 67.4
Разница (лицей − обычная): -0.7


**Вывод:** разница отрицательная (−0.7) — лицеисты в среднем сдают чуть хуже обычной школы, вопреки ожиданию; тип школы сам по себе балл не гарантирует.


### <font color='DarkOrange'>Задание 11 [2 балла]</font>

**Сравнение: высокая vs низкая мотивация.** Возьмите тех, у кого `мотивация_1_10 >= 8`,
и тех, у кого `мотивация_1_10 <= 3`. Посчитайте средний балл ЕГЭ в каждой группе.
**На сколько баллов** высокомотивированные опережают низкомотивированных?
В ответ запишите **целую часть** числа.


In [30]:
hi = df[df['мотивация_1_10'] >= 8]['балл_егэ'].mean()
lo = df[df['мотивация_1_10'] <= 3]['балл_егэ'].mean()
print('Мотивация >= 8:', round(hi, 1))
print('Мотивация <= 3:', round(lo, 1))
print('Разница       :', int(hi - lo))
# Ответ: 28


Мотивация >= 8: 79.7
Мотивация <= 3: 51.6
Разница       : 28


### <font color='DarkOrange'>Задание 12 [4 балла]</font>

Обучите линейную регрессию `LinearRegression` **на двух признаках сразу**:
`средний_балл_года` и `пропусков_уроков` — для предсказания `балл_егэ`.

**Какой балл ЕГЭ** модель предскажет ученику, у которого
**средний балл года = 3.5** и **пропусков уроков = 5**? Округлите до **целого числа**.


In [31]:
from sklearn.linear_model import LinearRegression

X = df[['средний_балл_года', 'пропусков_уроков']]
y = df['балл_егэ']
model = LinearRegression().fit(X, y)

new = pd.DataFrame({'средний_балл_года': [3.5], 'пропусков_уроков': [5]})
print(round(model.predict(new)[0]))
# Ответ: 77


77


**Вывод:** модель предсказывает 77 баллов — довольно оптимистично для середнячка, но именно такую связь она увидела в данных.
